# Ch 31 부록 — GRPO를 직접 망가뜨리고 고쳐보기: 학습 신호의 분산

> 본 챕터(Ch 31)에서 GRPO가 잘 작동하는 레시피를 봤습니다. 이 부록은 *왜 그 레시피가 필요했는지* 를 거꾸로, **직접 실패를 겪으며** 따라갑니다. 본 챕터와 **똑같은 설정**(SFT·난이도 필터·GRPOConfig·seed)을 그대로 쓰되, 거기에 "난이도 필터 없이 순진하게 돌린" 대조군을 하나 더 두어 *필터가 있고 없고의 차이* 를 직접 비교합니다.

**이 부록에서 직접 경험할 것**
1. 🧮 **advantage가 0이 되는 순간** — 그룹 보상이 전부 같으면 학습 신호가 사라지는 걸 손계산으로 확인
2. 💥 **순진한 GRPO(필터 없이)를 돌려 하락을 겪기** — 정확도가 안 오르거나 떨어지는 걸 눈으로 봄
3. 🔬 **진단** — 학습 문제의 정답률이 0/1로 *양극화(std=0)* 돼 있음을 출력
4. 🛠️ **난이도 필터로 수정** — 같은 pool에서 정답·오답이 섞이는 문제만 남겨 개선 확인 (본 챕터 결과 재현)

**환경**: Google Colab **T4 GPU**. **예상 소요**: 약 22분 (SFT + GRPO 2회 + 난이도 측정).

> ⚠️ 순진한 GRPO와 난이도 필터 GRPO는 **같은 SFT 출발점**에서 시작하도록, SFT 모델을 저장해 두고 매번 재로드합니다(공정 비교). 차이는 오직 *학습셋에 난이도 필터를 적용했는가* 하나입니다.

## 0. 환경 셋업

In [ ]:
%pip install -q -U trl transformers tokenizers datasets accelerate

import warnings, re, time, random, numpy as np, torch
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
print("device:", device, "| fp16:", USE_FP16)

## 1. 모델·토크나이저 (KoGPT2 125M) — 본 챕터와 동일

In [ ]:
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token="</s>", eos_token="</s>", unk_token="<unk>",
    pad_token="<pad>", mask_token="<mask>",
)
policy = AutoModelForCausalLM.from_pretrained("skt/kogpt2-base-v2").to(device)
policy.config.pad_token_id = tokenizer.pad_token_id
print("params:", f"{policy.num_parameters()/1e6:.1f}M")

## 2. 산술 태스크 + verifier — 본 챕터와 동일 (같은 seed)

한 자리 산술(`3 + 5 = ?`)을 풀게 합니다. 정답이 정해져 있어 채점이 자동입니다(verifiable reward). 데이터 seed도 본 챕터와 똑같이 맞춰 결과가 재현되게 합니다.

In [ ]:
from datasets import Dataset

RESPONSE_TEMPLATE = "### 응답:\n"
def build_prompt(q): return f"### 명령어:\n{q}\n\n{RESPONSE_TEMPLATE}"

def make_arithmetic(n, max_operand=9, seed=0):
    rng = random.Random(seed); rows = []
    for _ in range(n):
        a = rng.randint(1, max_operand); b = rng.randint(1, max_operand)
        op = rng.choice(["+", "-"]); ans = a + b if op == "+" else a - b
        rows.append({"prompt": build_prompt(f"{a} {op} {b} = ?"), "answer": str(ans)})
    return Dataset.from_list(rows)

def extract_answer(text):
    seg = text.split("###")[0]                 # 응답 블록만 (다음 문제 숫자 안 집게)
    m = re.search(r"-?\d+", seg)
    return m.group(0) if m else None

def reward_correct(completions, answer, **kw):  # 이진 verifier: 정답=1, 오답=0
    return [1.0 if (extract_answer(c) == str(g)) else 0.0
            for c, g in zip(completions, answer)]

eval_ds = make_arithmetic(64, seed=SEED + 1)   # 본 챕터와 동일

@torch.no_grad()
def eval_accuracy(model, dataset, n=64, max_new=24):
    # greedy(do_sample=False) 로 측정 고정 — 본 챕터와 동일
    model.eval(); correct = 0
    for ex in dataset.select(range(min(n, len(dataset)))):
        enc = tokenizer(ex["prompt"], return_tensors="pt").to(device)
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, num_beams=1,
                             pad_token_id=tokenizer.pad_token_id)
        txt = tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        correct += int(extract_answer(txt) == str(ex["answer"]))
    return correct / min(n, len(dataset))

print("base 모델 정확도:", round(eval_accuracy(policy, eval_ds), 3), "(거의 0 — 산술을 못 풉니다)")

## 3. SFT 워밍스타트 — 본 챕터와 동일 (5 epoch / 3000 샘플)

GRPO는 *없던 능력을 새로 만들지 못합니다.* base 모델은 산술 정확도 0%라, GRPO를 걸 *재료*(가끔 맞는 답)가 없습니다. 본 챕터와 똑같이 산술 포맷+정답을 지도학습(SFT)해 정확도를 끌어올립니다. 이 SFT 모델을 디스크에 저장해, 뒤의 두 GRPO 실험이 *같은 출발점* 에서 시작하게 합니다.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

sft_ds = make_arithmetic(3000, seed=SEED + 7)   # 본 챕터와 동일
def _to_sft(ex):
    return tokenizer(ex["prompt"] + ex["answer"] + tokenizer.eos_token,
                     truncation=True, max_length=48, padding="max_length")
sft_tok = sft_ds.map(_to_sft, remove_columns=sft_ds.column_names)

sft_args = TrainingArguments(output_dir="./out_grpo_sft", num_train_epochs=5,
    per_device_train_batch_size=32, learning_rate=5e-4, warmup_ratio=0.1,
    lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16, logging_steps=50,
    save_strategy="no", report_to="none")
Trainer(model=policy, args=sft_args, train_dataset=sft_tok,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)).train()

policy.save_pretrained("./sft_ckpt"); tokenizer.save_pretrained("./sft_ckpt")
acc_sft = eval_accuracy(policy, eval_ds)
print(f"\nSFT 후 정확도(greedy): {acc_sft:.3f}   <- GRPO 의 출발점 (본 챕터 ~0.875)")

## 4. 🧮 체험 1 — advantage가 0이 되는 순간 (손계산)

GRPO는 critic 없이, 한 prompt에 여러 답(그룹)을 생성해 **그룹 안에서** 잘한 답의 확률을 올립니다. 그 신호가 advantage 입니다.

$$A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r) + \varepsilon}$$

학습을 돌리기 전에, 그룹 보상이 어떤 모양일 때 학습 신호가 생기는지 *손으로* 봅시다.

In [ ]:
def group_advantage(rewards, eps=1e-4):
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

print(f"{'group reward':<22}{'std':>7}{'advantage':>26}   학습 신호")
print("-" * 70)
for rw in ([0,0,0,0], [1,1,1,1], [1,0,1,0]):
    adv = group_advantage(rw)
    sig = "있음" if np.std(rw) > 0 else "없음 (전부 같음)"
    print(f"{str(rw):<22}{np.std(rw):>7.2f}{str(np.round(adv,2)):>26}   {sig}")

**무엇을 봤나.** 그룹이 *전부 오답* `[0,0,0,0]` 이거나 *전부 정답* `[1,1,1,1]` 이면 std=0 → **advantage가 전부 0 → gradient 0 → 아무것도 안 배웁니다.** 오직 *정답·오답이 섞인* `[1,0,1,0]` 그룹에서만 학습 신호가 생깁니다. 이 한 장이 이 부록의 전부입니다 — 나머지는 이걸 실제로 확인하는 과정입니다.

## 5. 💥 체험 2 — 순진한 GRPO를 돌려보자 (난이도 필터 없이)

본 챕터와 똑같은 `GRPOConfig` 로, 다만 *학습셋에 난이도 필터를 적용하지 않고* 그대로 GRPO를 겁니다. "보상이 있으니 오르겠지"라는 순진한 기대로요. SFT 체크포인트에서 시작합니다.

학습셋은 본 챕터의 난이도 필터가 *고르기 전* 의 pool(500문제)에서 앞쪽 256개를 그대로 씁니다 — 뒤에서 이 pool에 필터를 적용해 비교합니다.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

def fresh_sft_model():
    m = AutoModelForCausalLM.from_pretrained("./sft_ckpt").to(device)
    m.config.pad_token_id = tokenizer.pad_token_id
    return m

GROUP_SIZE = 8
def grpo_cfg(tag):   # 본 챕터 GRPOConfig 와 동일
    return GRPOConfig(output_dir=f"./out_{tag}", num_train_epochs=2,
        per_device_train_batch_size=16, gradient_accumulation_steps=8,
        num_generations=GROUP_SIZE, max_completion_length=24, mask_truncated_completions=True,
        temperature=0.7, top_p=0.95, scale_rewards=False, loss_type="dr_grpo",
        learning_rate=5e-6, lr_scheduler_type="constant_with_warmup", warmup_ratio=0.1,
        max_grad_norm=0.2, beta=0.04, fp16=USE_FP16, logging_steps=10,
        save_strategy="no", report_to="none", use_vllm=False, seed=SEED)

pool = make_arithmetic(500, seed=SEED + 3)        # 본 챕터 난이도 필터와 같은 pool·seed
naive_ds = pool.select(range(256))                # 필터 없이 그대로

policy_naive = fresh_sft_model()
GRPOTrainer(model=policy_naive, reward_funcs=reward_correct,
            args=grpo_cfg("naive"), train_dataset=naive_ds,
            processing_class=tokenizer).train()

acc_naive = eval_accuracy(policy_naive, eval_ds)
del policy_naive; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"\nSFT 시작점     : {acc_sft:.3f}")
print(f"순진한 GRPO 후 : {acc_naive:.3f}   (Δ {acc_naive - acc_sft:+.3f})")
print("→ 보상이 있어도 정확도가 오르지 않거나 오히려 떨어집니다.")

## 6. 🔬 진단 — 학습 문제의 정답률은 0/1로 양극화돼 있다

왜 안 올랐을까요? 체험 1의 가설("그룹이 전부 같으면 신호 0")이 실제 데이터에서 사실인지 확인합니다. pool의 각 문제에 SFT 모델로 8개 답을 생성해 *정답률(pass rate)* 을 재고, 그 분포를 봅니다. (이 측정값은 다음 셀의 난이도 필터에도 그대로 씁니다.)

In [ ]:
@torch.no_grad()
def pass_rate_of(model, ex, k=8):
    enc = tokenizer(ex["prompt"], return_tensors="pt").to(device)
    gen = model.generate(**enc, max_new_tokens=24, do_sample=True, temperature=0.7, top_p=0.95,
                         num_return_sequences=k, pad_token_id=tokenizer.pad_token_id)
    return sum(extract_answer(tokenizer.decode(g[enc["input_ids"].shape[1]:],
               skip_special_tokens=True)) == str(ex["answer"]) for g in gen) / k

sft_probe = fresh_sft_model()
rates = np.array([pass_rate_of(sft_probe, ex) for ex in pool])   # pool 500 전체
del sft_probe; torch.cuda.empty_cache() if torch.cuda.is_available() else None

zero_std = np.mean((rates == 0.0) | (rates == 1.0))   # 전부오답 or 전부정답 = std 0
print("문제별 정답률(8개 중 맞은 비율) 분포:")
edges = [0.0, 0.125, 0.375, 0.625, 0.875, 1.001]
labels = ["0 (전부 오답)", "낮음", "중간", "높음", "1 (전부 정답)"]
for lo, hi, lab in zip(edges[:-1], edges[1:], labels):
    n = int(np.sum((rates >= lo) & (rates < hi)))
    print(f"  {lab:<14} {'#'*(n//5)} {n}")
print(f"\n→ std=0 인 그룹(전부 0 또는 전부 1) 비율: {zero_std:.0%}")
print("  이 문제들은 GRPO 에 *학습 신호를 전혀 주지 못합니다.* 순진한 학습셋의 상당수가 여기 해당합니다.")

## 7. 🛠️ 수정 — 난이도 필터로 std>0 그룹만 남기기 (본 챕터와 동일)

진단이 맞다면, 해결책은 단순합니다. **같은 pool에서**, 그룹 안에 정답과 오답이 섞이는 문제(정답률이 0도 1도 아닌, 중간 난이도)만 골라 GRPO 학습셋으로 씁니다. 보상은 그대로(이진), 데이터만 거릅니다 — 본 챕터가 한 그대로입니다.

In [ ]:
keep_idx = [i for i, r in enumerate(rates) if 0.25 <= r <= 0.875]   # 중간 난이도
filtered_ds = pool.select(keep_idx[:256])
print(f"pool {len(pool)}개 -> 중간 난이도 {len(filtered_ds)}개 (그룹에 정답·오답 섞임 = std>0)")

## 8. ✅ 체험 3 — 수정 후 GRPO (본 챕터 결과 재현)

같은 `GRPOConfig`, 같은 SFT 출발점, 같은 학습량. **딱 하나, 학습셋만 난이도 필터를 통과한 것**으로 바꿉니다. 이제 GRPO가 배울 신호가 있습니다 — 본 챕터의 결과(SFT → GRPO 소폭 상승)가 재현되는지 봅니다.

In [ ]:
policy_fixed = fresh_sft_model()
GRPOTrainer(model=policy_fixed, reward_funcs=reward_correct,
            args=grpo_cfg("fixed"), train_dataset=filtered_ds,
            processing_class=tokenizer).train()

acc_fixed = eval_accuracy(policy_fixed, eval_ds)
print("\n=== 세 줄 요약 ===")
print(f"SFT 시작점       : {acc_sft:.3f}")
print(f"순진한 GRPO      : {acc_naive:.3f}   (Δ {acc_naive - acc_sft:+.3f})  <- 필터 없음(신호 부족)")
print(f"난이도 필터 GRPO : {acc_fixed:.3f}   (Δ {acc_fixed - acc_sft:+.3f})  <- 필터 있음(신호 생김)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["SFT start", "naive GRPO", "filtered GRPO"],
              [acc_sft, acc_naive, acc_fixed],
              color=["tab:gray", "tab:red", "tab:green"], alpha=0.85)
for b, v in zip(bars, [acc_sft, acc_naive, acc_fixed]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}", ha="center")
ax.set_ylabel("arithmetic accuracy (greedy)")
ax.set_title("GRPO: naive (no filter) vs difficulty-filtered")
ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

## 9. 정리 — 무엇을 배웠나

직접 겪은 순서대로:

1. **체험 1 (손계산)**: 그룹 보상이 전부 같으면 advantage=0 → 학습 신호 없음.
2. **체험 2 (순진한 GRPO)**: 본 챕터와 똑같은 설정인데 *필터만 빼니* 정확도가 안 오름/하락.
3. **진단**: 한 자리 산술은 문제별 정답률이 0 또는 1로 *양극화* 돼, pool의 상당수가 std=0.
4. **수정 (난이도 필터)**: 같은 pool에서 정답·오답이 섞이는 문제만 남기니 GRPO가 (+) — 본 챕터 결과 재현.

**핵심 교훈**
- GRPO에서 "보상이 있다 ≠ 학습이 된다." **보상의 *차이*(분산)** 가 있어야 배웁니다.
- 그래서 *무엇을 보상하는가* 만큼 *어떤 문제로 학습하는가* 가 중요합니다. 너무 쉽거나 너무 어려운 문제는 신호를 못 줍니다.
- 본 챕터(Ch 31)의 난이도 필터는 바로 이 *신호 생성* 장치였습니다. 이 부록은 그 *이유* 를 손으로 만져 본 것입니다.